# Step 1: Preparation — Load libraries

In [1]:
!python -m spacy download en_core_web_lg

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 400.7/400.7 MB 2.9 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_lg')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
import csv
import unicodedata
import pandas as pd
from collections import Counter
from itertools import combinations
import matplotlib.pyplot as plt
import seaborn as sns
import networkx as nx
import spacy
from spacy.lang.en import English
from spacy.lang.en.stop_words import STOP_WORDS
import re

nlp = spacy.load('en_core_web_lg') # Load the large spaCy NLP model
nlp.max_length = 1200000 # Increase max length to fit the text and avoid errors later for a long text

# Step 2: Preprocess the text

In [3]:
# Basic cleaning function

def clean_text(text):
    if text is None:
        raise ValueError("Input text is None.")

    # Replace newline, carriage return, and tab characters with a space
    text = re.sub(r'[\n\r\t]', ' ', text)

    # Normalise text to remove encoding artifacts
    text = unicodedata.normalize('NFKD', text)

    # Replace problematic encoding artifacts
    text = text.replace('â', '').replace('€', '').replace('_', '').replace('™', '').replace('œ', '')

    return text

# Advanced cleaning function
def clean_text_advanced(text):
    """
    Further cleans the text by:
    - Removing possessive "'s"
    - Removing punctuation, symbols, and numbers
    - Removing stopwords
    """
    if text is None:
        raise ValueError("Input text is None.")

    # Remove possessive "'s"
    text = re.sub(r"\b(\w+)'s\b", r"\1", text)

   # Remove some punctuation and symbols except end punctuation
    text = re.sub(r"[^\w\s.!?]", " ", text)

    # Remove numbers
    text = re.sub(r"\d+", " ", text)

    # Tokenize the text using spaCy
    doc = nlp(text)

    # Remove stopwords and extra spaces
    cleaned_tokens = [token.text for token in doc if token.text.lower() not in STOP_WORDS]
    cleaned_text = " ".join(cleaned_tokens)

    return cleaned_text

# Process the text file
def process_file(file_path, output_file_path=None):

    try:
        # Read the file
        with open(file_path, 'r', encoding='utf-8') as f:
            text = f.read()
        print(f"File read successfully: {file_path}")

        # Run basic cleaning
        text = clean_text(text)
        print("Basic cleaning completed.")

        # Run advanced cleaning
        text = clean_text_advanced(text)
        print("Advanced cleaning completed.")

        # Save cleaned text
        save_path = output_file_path if output_file_path else file_path
        with open(save_path, 'w', encoding='utf-8') as f:
            f.write(text)
        print(f"Text cleaned and saved successfully: {save_path}")
    except FileNotFoundError:
        print(f"File not found: {file_path}")
    except ValueError as e:
        print(f"Error during processing: {e}")

# Input file path
file_path = r'./The_Mill_on_the_Floss.txt'

# Output file path
output_file_path = r'./The_Mill_on_the_Floss_cleaned.txt'

# Process the file and save with a new name
process_file(file_path, output_file_path)

# Step 3: Split the text into sentences

In [4]:
# Configure function with a suitable max_length for the text
sentencizer = English()
sentencizer.max_length = 1_200_000
sentencizer.add_pipe("sentencizer")

# Read the cleaned text content from the file
with open(output_file_path, 'r', encoding='utf-8') as f:
 text = f.read()

# Split text into sentences
sentences = [sent.text for sent in sentencizer(text).sents]

# Print first 10 sentences to see results
print(sentences[:10])

# Step 4: Load and run the NER model

In [5]:
# Create a function to remove encoding artifacts that can cause a problem later
def normalize_text(text):
    return unicodedata.normalize('NFKD', text)

# Start the NER process and save the results
with open('./ner_results.csv', 'w', newline='', encoding='utf-8') as f:
    writer = csv.writer(f)
    writer.writerow(['Sentence', 'Entity', 'Label'])

    for doc in nlp.pipe(sentences, batch_size=50):  # Process sentences in batches
        for ent in doc.ents:
           writer.writerow([normalize_text(doc.text), normalize_text(ent.text), ent.label_]) # Normalise text before saving the file

# Step 5: Extract character names and their frequency

In [6]:
df = pd.read_csv('./pathway/ner_results.csv')

character_counts = df[df['Label'] == 'PERSON']['Entity'].value_counts()

character_counts = character_counts.reset_index()

character_counts.columns = ['Name', 'Count']

# Show the top 10 characters
print(character_counts.head(10))

# Save to CSV
character_counts.to_csv('./pathway/character_counts.csv', index=False)

# Step 6: Plot of characters with seaborn

In [7]:
# Load character counts
character_counts = pd.read_csv('./pathway/character_counts.csv')

# Select the top 15 characters
top_characters = character_counts.nlargest(15, 'Count')

# Set the plot size and aesthetic style
sns.set_theme(style="whitegrid")
plt.figure(figsize=(15, 10))
sns.barplot(x='Name', y='Count', data=top_characters, palette='colorblind')

# Rotate x-axis labels for better readability
plt.xticks(rotation=45, ha='right', fontsize=12)

# Add titles and labels
plt.title('Top 15 characters in The Mill on the Floss', fontsize=18, fontweight='bold')
plt.xlabel('Character name', fontsize=14, fontweight='bold')
plt.ylabel('Frequency', fontsize=14, fontweight='bold')

plt.show()

# Save the plot to a file in the interface or with:
plt.savefig('./pathway/top_characters.png', bbox_inches='tight')

# Step 7: Analyse main character relationships

In [8]:
sentence_entities = df.groupby('Sentence')['Entity'].apply(list)

# Find characters mentioned together
co_occurrences = []
for entities in sentence_entities:
    if len(entities) >= 2:
        co_occurrences.extend(combinations(set(entities), 2))

# Show the 15 most common character pairs
print("15 most frequent character pairs:")
print(Counter(co_occurrences).most_common(15))

# Convert pairings to a DataFrame
co_occurrences_df = pd.DataFrame(Counter(co_occurrences).most_common(15), columns=['Pair', 'Count'])

# Save all co_occurrences
co_occurrences_df.to_csv('./character_co_occurrences_top_15_pairs.csv', index=False)

In [9]:
sentence_entities = df.groupby('Sentence')['Entity'].apply(list)

# Find groups of three characters mentioned together
co_occurrences_3 = []
for entities in sentence_entities:
    if len(entities) >= 3:
        co_occurrences_3.extend(combinations(set(entities), 3))

# Show the 15 most common triads
print("15 most frequent character triads:")
print(Counter(co_occurrences_3).most_common(5))

# Convert all co_occurrences to a DataFrame for saving
co_occurrences_3_df = pd.DataFrame(Counter(co_occurrences_3).most_common(15), columns=['Triad', 'Count'])

# Save all co_occurrences to CSV
co_occurrences_3_df.to_csv('./pathway/character_co_occurrences_top_15_triads.csv', index=False)